In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os, sys, json
import pandas as pd
import datetime as dt
from dateutil.relativedelta import relativedelta

from rockyelevate.wrapper import Session
from rockyelevate.utils import response_to_dataframe
from rockydb.connection import CoreDB

current_dir = os.getcwd()
parent_dir = os.path.dirname(f"{"\\".join(current_dir.split("\\")[:-1])}")
sys.path.append(parent_dir)

elv = Session("PROD", multithread=True, max_threads=40)
cdb = CoreDB()


In [ ]:
with open("_simple_plans_9d12d5a8_20251229.json", "r") as f:
    simple_plans_json = json.load(f)
    simple_plan_df = response_to_dataframe(simple_plans_json)

for col in ['plan_year.valid_from', 'plan_year.valid_to']:
    simple_plan_df[col] = pd.to_datetime(simple_plan_df[col])

print(f"{len(simple_plan_df)} simple plans")

simple_plan_df['plan_length'] = simple_plan_df.apply(lambda r: r['plan_year.valid_to'] - r['plan_year.valid_from'], axis=1)


In [ ]:
# find plans that end on dec 31st 2025
dec_31 = dt.datetime(2025, 12, 31)
dec_31_plans = simple_plan_df[simple_plan_df['plan_year.valid_to'] == dec_31]
print(f"{len(dec_31_plans)} plans end on {dec_31.strftime("%m/%d/%Y")}")

In [ ]:
# now we need to find other plans that have the same account type, organization id and a valid_from date soon after these plans valid_to's

rolled_plan_map = {}
for identifier, grouped_df in simple_plan_df.groupby(by=['organization_id', 'account_type']):

    p_2025 = grouped_df[grouped_df['plan_year.valid_to'] == dec_31]
    if p_2025.empty:
        continue # if there are no dec_31 plans
    p_2025 = p_2025.iloc[0]

    p_2026 = grouped_df[grouped_df['plan_year.valid_from'] > dec_31]
    if p_2026.empty:
        rolled_plan_map[p_2025['id']] = None
    else:
        rolled_plan_map[p_2025['id']] = p_2026.iloc[0]['id']

missing_rolls = dec_31_plans[dec_31_plans['id'].isin([k for k,v in rolled_plan_map.items() if pd.isna(v)])]
missing_rolls = missing_rolls[
    (missing_rolls['plan_length'].dt.days > 360) &  # drop short plans, am's can deal w them
    (missing_rolls['plan_length'].dt.days < 370) &
    ~(missing_rolls['plan_status'].isin(['PENDING_TERMINATION', 'TERMINATED', 'TERMINATION_IN_PROGRESS', 'INACTIVE']))
]

print(f"{len(missing_rolls)} dec_31 plans have not been rolled")


In [ ]:
# check to see if these plans appear in any active ('NEW') renewals
oids = list(missing_rolls['organization_id'].unique())
renewals_res = elv.get_renewals(oids)
renewal_df = response_to_dataframe(renewals_res)

renewal_status_map = {r.get("id"): r.get("status") for r in renewals_res}

plan_renewal_map = {}
for _, r in renewal_df.iterrows():
    for r_plan in r['plans']:
        plan_renewal_map[r_plan['prior_plan_id']] = r['id']

missing_rolls['renewal_id'] = missing_rolls['id'].map(plan_renewal_map)
missing_rolls['renewal_status'] = missing_rolls['renewal_id'].map(renewal_status_map)

plans_found_in_renewals = missing_rolls[missing_rolls['renewal_id'].notna()]

print(f"{len(plans_found_in_renewals)} plans found in {len(plans_found_in_renewals['renewal_id'].unique())} renewals")


In [ ]:
renewals_to_send = {}

for renewal_id, grouped_df in missing_rolls[missing_rolls['renewal_id'].notna()].groupby("renewal_id"):
    renewals_to_send[int(renewal_id)] = grouped_df['id'].to_list()

renewals_to_send


In [ ]:
import re
from copy import deepcopy


ELV_ACCOUNT_TYPE_MAP = {
    "DCAP": "DCA",
    "HCFSA": "FSA",
    "PARKING": "PKG",
    "TRANSIT": "TRN",
    "SPECIALTY": "LSA",
    "LIFESTYLE": "LSA",
    "ADOPTION": "ADO",
    "HRA": "HRA",
    "HSA": "HSA",
}


def update_plan_name(new_plan, valid_from):

    # get current plan name
    new_plan_name = new_plan.get("name")

    # finnd all numbers in plan name
    number_substrings = re.findall(r'\d+', new_plan_name)

    # loop through all numbers
    for num in number_substrings:
        # if the number is larger than `2020` remove it
        if int(num) > 2020:
            new_plan_name = new_plan_name.replace(num, "")

    # after removing the year from the name, check if there is another number (usually 2) at the end
    new_plan_name = new_plan_name.strip()
    number_substrings = re.findall(r'\d+', new_plan_name)
    # if so, remove it
    if number_substrings and new_plan_name[-len(number_substrings[-1])] == number_substrings[-1]:
        new_plan_name = new_plan_name.replace(number_substrings[-1], "")

    if "delete" in new_plan_name.lower():
        raise ValueError(f"'delete' in plan name {new_plan_name}")

    # remove `template` from the plan name
    for x in [y for y in [
        'template',
        'Template',
        "TEMPLATE",
    ] if y in new_plan_name]:
        new_plan_name = new_plan_name.replace(x, "")

    new_plan_name = new_plan_name.replace("  ", " ")

    # split the name on spaces and rejoin.
    # forgot why i did this, prolly to fix formatting issues
    name_split = new_plan_name.split(" ")
    name_split = [s for s in name_split if s != '']
    new_plan_name = " ".join(name_split)

    # return early if it's an hsa
    if new_plan.get("account_type") == "HSA":
        return new_plan_name

    # otherwise add the current plan year to the new plan name
    new_plan_name = f"{new_plan_name} {valid_from.year}"
    return new_plan_name


def get_rmrcode(organization_id):
    rmrcode = None

    entity = cdb.get_table_item_by_attribute(table="entity", attribute="elevate_id", value=organization_id)
    if entity:
        rmrcode = entity.employer_id

    else:
        org_res = elv.get_organizations(oids=[organization_id])
        rmrcode = org_res.get("external_identifier", None)

    return rmrcode


def format_plan(plan, rmrcode, valid_from, valid_to):
    new_plan = deepcopy(plan)

    new_plan['name'] = update_plan_name(plan, valid_from)

    account_type = ELV_ACCOUNT_TYPE_MAP.get(new_plan.get("account_type"))
    new_plan["plan_code"] = (
        f"{rmrcode}{account_type}{valid_from.strftime('%m%d%Y')}{valid_to.strftime('%m%d%Y')}"
    )
    

    return new_plan

In [ ]:

def format_renewal(renewal_body, plan_ids):
    org_id = renewal_body.get("organization_id")
    rmrcode = get_rmrcode(org_id)
    if not rmrcode:
        raise ValueError(f"No rmrcode found for organization ({org_id})!")

    new_renewal = deepcopy(renewal_body)
    new_plan_year = new_renewal.get("new_plan_year")

    valid_from = pd.to_datetime(new_plan_year.get("valid_from"))
    valid_to = pd.to_datetime(new_plan_year.get("valid_to"))

    if (valid_to - valid_from).days < 363:
        raise ValueError(f"renewal {new_renewal.get("id")} is too short ({(valid_to - valid_from).days} days)!")
    
    new_plan_year["name"] = f"{dt.datetime.strftime(valid_from, "%m/%d/%Y")} - {dt.datetime.strftime(valid_to, "%m/%d/%Y")}"

    plans_to_renew = [
        p for p in new_renewal.get("plans", [])
        if p.get("prior_plan_id") in plan_ids
    ]

    if not plans_to_renew:
        print(f"renewal {new_renewal.get("id")} has {len(plans_to_renew)}/{len(new_renewal.get("plans"))} plans to roll, skipping...")

    formatted_plans = [ format_plan(plan, rmrcode, valid_from, valid_to) for plan in plans_to_renew ]

    new_renewal['plans'] = formatted_plans

    renewal_json = {}
    for key in ["new_plan_year", "plans", "copays"]:
        renewal_json[key] = new_renewal[key]
    
    request_body = {}
    for key in ['id', 'plan_year_id', 'org_id']:
        request_body[key] = new_renewal[key]

    request_body['status'] = "ACTIVE"
    request_body['renewal_json'] = renewal_json

    return request_body


In [ ]:
renewal_requests = []

for renewal_id, plan_ids in renewals_to_send.items():
    renewal_body = [r for r in renewals_res if r.get("id") == renewal_id][0]
    formatted_renewal = format_renewal(renewal_body, plan_ids)
    renewal_requests.append(formatted_renewal)


In [ ]:
def activate_renewal(renewal_request):
    plan_year_id = renewal_request.get("plan_year_id")
    url = f"{elv.basepath}/renewals/plan-year/{plan_year_id}"

    elv_response = elv.patch(endpoint=url, payload=renewal_request)

    return elv_response

In [ ]:
# renewal_responses = []

# for renewal_request in renewal_requests:
#     res = activate_renewal(renewal_request)
#     renewal_responses.append(res)

# parsed_responses = [r.json() for r in renewal_responses]

# with open("renewal_responses.json", "w") as f:
#     json.dump(parsed_responses, f, indent=4)

with open("renewal_responses.json", "r") as f:
    parsed_responses = json.load(f)

renewal_status_map = {}
prior_plan_map = {}
for renewal in parsed_responses:
    for plan in renewal['renewal_json']['plans']:
        renewal_status_map[plan['prior_plan_id']] = renewal.get("status")
        prior_plan_map[plan['prior_plan_id']] = renewal.get("id")

missing_rolls['current_status'] = missing_rolls['id'].map(renewal_status_map)
missing_rolls['renewal_id'] = missing_rolls['id'].map(prior_plan_map)

In [ ]:
# check what plan years need to be made

plans_to_manually_roll = missing_rolls[missing_rolls['renewal_id'].isna()]
org_ids = list(plans_to_manually_roll["organization_id"].unique())
plan_years = elv.get_plan_years(oids=org_ids)
plan_year_df = response_to_dataframe(plan_years)

for col in ['valid_from', 'valid_to']:
    plan_year_df[col] = pd.to_datetime(plan_year_df[col])

plan_year_map = {}
for years, grouped_df in plans_to_manually_roll.groupby(by=['organization_id', 'plan_year.valid_from', 'plan_year.valid_to']):
    org_id, valid_from, valid_to = years

    new_valid_from = valid_from + relativedelta(years=1)
    new_valid_to = valid_to + relativedelta(years=1)

    new_plan_year = plan_year_df[
        (plan_year_df['valid_from'] == new_valid_from) &
        (plan_year_df['organization_id'] == org_id)
    ]

    if not new_plan_year.empty:
        plan_year_map[org_id] = new_plan_year.iloc[0]['id']

plans_to_manually_roll['new_plan_year_id'] = plans_to_manually_roll['organization_id'].map(plan_year_map)

In [ ]:
plan_years_to_create = plans_to_manually_roll[plans_to_manually_roll['new_plan_year_id'].isna()]

new_plan_year_bodies = []
for py, grouped_df in plan_years_to_create.groupby(by=["organization_id", "plan_year.valid_from", "plan_year.valid_to", "plan_year.id"]):
    org_id, valid_from, valid_to, prior_plan_year_id = py

    new_valid_from = (valid_from + relativedelta(years=1)).strftime("%m/%d/%Y")
    new_valid_to = (valid_to + relativedelta(years=1)).strftime("%m/%d/%Y")

    new_plan_year = {
        "organization_id": int(org_id),
        "name": f"{new_valid_from} - {new_valid_to}",
        "valid_from": new_valid_from,
        "valid_to": new_valid_to,
        "prior_plan_year_id": int(prior_plan_year_id)
    }

    new_plan_year_bodies.append(new_plan_year)

# display(new_plan_year_bodies)
print(f"{len(new_plan_year_bodies)} plan year(s) to be created")



In [ ]:
new_plan_year_responses = []
for new_plan_year in new_plan_year_bodies:
    url = f"{elv.basepath}/plan-years"

    res = elv.post(endpoint=url, payload=new_plan_year)

    new_plan_year_responses.append(res)


In [ ]:
elv.template_ids.get("PROD")

In [ ]:
ELV_ACCOUNT_TYPE_MAP = {
    "DCAP": "DCA",
    "HCFSA": "FSA",
    "PARKING": "PKG",
    "TRANSIT": "TRN",
    "SPECIALTY": "LSA",
    "LIFESTYLE": "LSA",
    "ADOPTION": "ADO",
    "HRA": "HRA",
    "HSA": "HSA",
}

def create_plan_name(row):
    new_plan_name = row.get("name")
    
    number_substrings = re.findall(r'\d+', new_plan_name)
    
    for num in number_substrings:
        if int(num) > 2020:
            new_plan_name = new_plan_name.replace(num, "")
    
    new_plan_name = new_plan_name.strip()
    number_substrings = re.findall(r'\d+', new_plan_name)
    if number_substrings and new_plan_name.endswith(number_substrings[-1]):
        new_plan_name = new_plan_name.replace(number_substrings[-1], "")
    
    if "delete" in new_plan_name.lower():
        raise ValueError(f"'delete' in plan name {new_plan_name}")
    
    for template_word in ['template', 'Template', "TEMPLATE"]:
        if template_word in new_plan_name:
            new_plan_name = new_plan_name.replace(template_word, "")
    
    new_plan_name = new_plan_name.replace("  ", " ")
    
    name_split = new_plan_name.split(" ")
    name_split = [s for s in name_split if s != '']
    new_plan_name = " ".join(name_split)
    
    if row.get("account_type") == "HSA":
        return new_plan_name
    
    new_plan_name = f"{new_plan_name} 2026"
    return new_plan_name


def create_plan_code(row, rmrcode):
    rmrcode = get_rmrcode(row['organization_id'])
    return f"{rmrcode}{ELV_ACCOUNT_TYPE_MAP.get(row['account_type'])}{(row.get("plan_year.valid_from") + relativedelta(years = 1)).strftime("%m%d%Y")}{(row.get("plan_year.valid_to") + relativedelta(years = 1)).strftime("%m%d%Y")}"


In [ ]:
print(len(plans_to_manually_roll))

naked_bodies = []

for organization_id, grouped_plans in plans_to_manually_roll.groupby("organization_id"):
    rmrcode = get_rmrcode(organization_id)

    for _, row in grouped_plans.iterrows():
        new_name = create_plan_name(row)
        new_plan_code = create_plan_code(row, rmrcode)

        new_plan = {
            "plan_year_id": row['new_plan_year_id'],
            "organization_id": row['organization_id'],
            "parent_id": elv.template_ids['PROD'].get(row['account_type']),
            "plan_code": new_plan_code,
            "name": {
                "name": new_name,
                "name_state": "MODIFIABLE"
            },
            "prior_plan_id": row.get("id"),
            "is_plan": True
        }

        naked_bodies.append(new_plan)


In [ ]:
def post_naked_body(naked_body):
    url = f"{elv.basepath}/plans"
    response = elv.post(endpoint=url, payload=naked_body)
    return response


In [ ]:
post_responses = []
for naked_body in naked_bodies:
    res = post_naked_body(naked_body)
    post_responses.append(res)

In [ ]:
from ast import literal_eval

error_response_map = {}
for post_response in post_responses:
    if isinstance(post_response, tuple):

        request_body = post_response[1].request.body.decode()
        prior_plan_id_index = request_body.find("prior_plan_id")
        prior_plan_id = request_body[prior_plan_id_index+16:].split(",")[0]

        error_response_map[int(prior_plan_id)] = post_response[1].json()['elevate_error_message']
    else:

        prior_plan_idx = post_response.json().get("prior_plan_id")
        error_response_map[prior_plan_idx] = f"new plan id: {post_response.json().get("id")}"


In [ ]:
plans_to_manually_roll['manual_create_response'] = plans_to_manually_roll['id'].map(error_response_map)

In [ ]:
plans_created = plans_to_manually_roll[plans_to_manually_roll['manual_create_response'].str.contains("new plan id")]

update_bodies = []
for index, row in plans_created.iterrows():
    new_plan_id = row['manual_create_response'].split(": ")[1]

    print(f"{row['id']}: {new_plan_id}")

    prior_plan_res = elv.get_plans_by_id(pids=[int(row['id'])])[0]
    new_plan_res = elv.get_plans_by_id(pids=[int(new_plan_id)])[0]

    for key in [k for k in [
        "parent_id",
        "plan_omnibus_account_id",
        "notional_payroll_account_id",
        "notional_funding_account_id",
        "plan_year",
        "service_configs",
        "prior_plan",
    ] if k in new_plan_res]:
        new_plan_res.pop(key, None)

    for f in [
        "plan_primary_config",
        "plan_coverage_config",
        "plan_account_funding_config",
    ]:
        new_plan_res[f] = prior_plan_res[f]

    new_plan_res['plan_primary_config'].pop("funding_method_type")

    fields_to_set = {
        "plan_status": "DRAFT",
    }

    for f in fields_to_set.keys():
        new_plan_res[f] = fields_to_set[f]


    update_bodies.append(new_plan_res)

In [ ]:
update_responses = []

for update_body in update_bodies:
    detailed_elv_res = elv.put(
        endpoint=f"{elv.basepath}/plans/{update_body['id']}",
        payload=update_body
    )
    update_responses.append(detailed_elv_res)


In [ ]:
update_json = [u.json() for u in update_responses]

update_map = {u.get("prior_plan_id"): "UPDATED" for u in update_json}


In [ ]:
plans_to_manually_roll['manual_update_response'] = plans_to_manually_roll['id'].map(update_map)

In [ ]:
missing_rows = missing_rolls.loc[[i for i in missing_rolls.index if i not in plans_to_manually_roll.index]]

In [ ]:
merge_df = pd.concat([plans_to_manually_roll, missing_rows])

In [ ]:
# merge_df.to_csv("FINAL_ROLL.csv")
print(len(merge_df))

In [ ]:
display(merge_df[merge_df['manual_create_response'].apply(lambda x: pd.notna(x) and "new plan id" in x)])